# CarbonGlobe — global-grid training · LSTM-XL backbone · 2×T4 · resumable

**IEEE Big Data Cup 2026 — CarbonGlobe AI Emulation Challenge**

All earlier runs trained on the 3,373 sites of `res_train4_test8.npz` (6% of the grid). The same Kaggle dataset
also ships the full simulated grid, `data_global/glob_X_fea.npy` + `glob_Y{age}.npy`: **54,152 sites × 40 years ×
15 seed ages**. This notebook trains on about 51.6k of them (roughly 16× more trajectories) with the fusion-lite engine
(MirroredStrategy, `tf.while_loop` rollout, scheduled sampling, EMA, checkpoint/resume).

| | fusion-lite | this notebook |
|---|---|---|
| training sites | 3,373 | all usable grid sites minus the holdout (~51.6k) |
| validation | 852 sites of `x_test` | two holdouts from the grid: **shift** (1,500 sites most shifted toward the test climate) and **random** (1,000) |
| default backbone | fusion, lite size | `lstm_xl` at the LB-0.168 size (`PRESET='full'`) |
| clamp at 0 during training | zero gradient below the floor | straight-through (`CLAMP_STE`) |
| input format | as stored | as stored by default; optional `HARMONIZE=True` (off: shown not to matter): features the test only gives as a
  repeated monthly climatology or without a seasonal cycle are given the same form in training |
| outputs | `submission.csv` | `submission.csv`, `submission_clip.csv`, `submission_co2_hist.csv` (see section 11) |

Memory: the forcing is kept standardized in float16 (~7.1 GB) plus the annual states (~0.9 GB). Loading reads about
25 GB from the dataset, which takes a few minutes and is counted in the session budget.

## How to run it on Kaggle (multi-session)

1. **Settings**: Accelerator = **GPU T4 x2**, Internet = on (for `git clone`). Inputs: the competition data and the
   `zhihaow/carbonglobe` dataset. If the repo is private, add the Kaggle Secret `GITHUB_TOKEN`.
2. **Session 1**: *Save Version → Save & Run All (Commit)*. Training stops about `SESSION_LIMIT_H − FINAL_RESERVE_MIN`
   after start, then writes `submission.csv`, `submission_clip.csv`, `global_lstmxl_preds.npz`,
   `global_lstmxl_holdout.npz` and the checkpoint folder **`global_lstmxl_ckpt/`**.
3. **Next sessions**: attach that output as an input (e.g. create a dataset from the *Output* tab, add a new version
   each session) and run again. The newest `global_lstmxl_ckpt/state.json` under `/kaggle/input` is picked up and
   training continues at the next epoch with the same weights, EMA, optimizer moments, LR schedule and counters.
4. Keep `PRESET`, `BACKBONE` and the holdout settings (`HOLDOUT_*`, `SUS_FEATS`, `SEED`) unchanged across sessions:
   the checkpoint refuses to load otherwise. LR, curriculum, time budget and batch size may change.

## 0. Clone repo & imports

In [1]:
import time
NB_T0 = time.time()   # session clock: the time budget is measured from here

import os, sys, subprocess, time, gc

REPO_DIR = '/kaggle/working/carbon-globe'
REPO_URL = 'github.com/kuongan/carbon-globe.git'

def _github_token():
    # Store the token in a Kaggle Secret named GITHUB_TOKEN (Add-ons -> Secrets).
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret('GITHUB_TOKEN')
    except Exception:
        return os.environ.get('GITHUB_TOKEN', '')

if not os.path.isdir(REPO_DIR):
    tok = _github_token()
    url = f'https://{tok}@{REPO_URL}' if tok else f'https://{REPO_URL}'
    r = subprocess.run(['git', 'clone', '--depth', '1', url, REPO_DIR], capture_output=True, text=True)
    if r.returncode != 0:
        err = r.stderr.replace(tok, '***') if tok else r.stderr
        raise RuntimeError('git clone failed:\n' + err)

PKG_DIR = os.path.join(REPO_DIR, 'LSTM-LSTNet-DeepED')
sys.path.insert(0, PKG_DIR)
os.chdir(PKG_DIR)
print('cwd =', os.getcwd())

cwd = /kaggle/working/carbon-globe/LSTM-LSTNet-DeepED


In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tqdm.auto import tqdm
import json, glob, shutil, zlib

from configs.constant_glob import N_CONS, N_OUT, N_YEAR, N_T, N_FEA, TREE_BAND
from utils.func import normData, invNormData

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)
print('TF', tf.__version__, '| Keras', keras.__version__)
print('N_CONS=%d N_OUT=%d N_FEA=%d N_YEAR=%d N_T=%d' % (N_CONS, N_OUT, N_FEA, N_YEAR, N_T))
for g in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(g, True)
    except Exception as e: print('memory growth note:', e)

TF 2.20.0 | Keras 3.13.2
N_CONS=136 N_OUT=7 N_FEA=158 N_YEAR=40 N_T=12


## 1. Constants, stats & competition metric

In [3]:
COMP = '/kaggle/input/competitions/ieee-bigdata-cup-2026-ai-emulation-challenge/public'
BASE = '/kaggle/input/datasets/zhihaow/carbonglobe'
OUT  = '/kaggle/working'

SEED_AGES = [1, 10, 20, 30, 50, 70, 100, 150, 200, 250, 300, 350, 400, 450, 500]
SCENARIOS = ['ssp126', 'ssp585']
N_AGES, N_AGE_FEA = len(SEED_AGES), 15

stats = np.load(f'{BASE}/data_stats/data_stats.npz')
X_MEAN, X_STD = stats['x_mean'].astype(np.float32), stats['x_std'].astype(np.float32)
Y_MEAN, Y_STD = stats['y_mean'].astype(np.float32), stats['y_std'].astype(np.float32)

agetri = np.load(f'{BASE}/data_stats/age_triplet.npz')
TRI_AGE   = agetri['age'].astype(np.float32)
TRI_MEAN  = agetri['age_mean'].astype(np.float32)
TRI_SLOPE = agetri['age_slope'].astype(np.float32)
assert list(TRI_AGE.astype(int)) == SEED_AGES

AGE_MEAN_N  = normData(TRI_MEAN,  Y_MEAN, Y_STD)
AGE_SLOPE_N = normData(TRI_SLOPE, Y_MEAN, Y_STD)
AGE_FEAT = np.stack([np.concatenate([[TRI_AGE[i] / 500.0], AGE_MEAN_N[i], AGE_SLOPE_N[i]])
                     for i in range(N_AGES)]).astype(np.float32)
FLOOR_N = normData(np.zeros(N_OUT, np.float32), Y_MEAN, Y_STD).astype(np.float32)
print('Y_MEAN =', np.round(Y_MEAN, 5))
print('submission scale -> height /%.5f , agb /%.5f' % (Y_MEAN[0], Y_MEAN[1]))


def scored_metric(pred, true):
    """The competition metric on year-40 height & AGB (physical units in, scaled MSE out)."""
    ph, th = pred[..., 0] / Y_MEAN[0], true[..., 0] / Y_MEAN[0]
    pa, ta = pred[..., 1] / Y_MEAN[1], true[..., 1] / Y_MEAN[1]
    mse_h = float(np.mean((ph - th) ** 2)); mse_a = float(np.mean((pa - ta) ** 2))
    return 0.5 * (mse_h + mse_a), mse_h, mse_a

Y_MEAN = [10.90644  3.39567  6.24214  1.88269  1.081    0.52339  0.489  ]
submission scale -> height /10.90645 , agb /3.39567


## 2. Config

In [4]:
# ---- run profile (see PLAN.md runbook) -------------------------------------------------------------------------
# CO2_MODE 'as_is' : the CO2 block (features 8-31) is a normal input (runs 1-2, LB 0.122 with co2_hist)
#          'const' : the CO2 block is set to its grid mean (0 after standardization) in training, holdout and test,
#                    so the model cannot learn a CO2 response; scenario effects are left to the post-hoc factors
CO2_MODE = 'const'
RUN_NAME = 'global_noco2_s7'     # session B was 'global_noco2' (SEED 42); use a new name per profile, e.g. 'global_s7' for CO2_MODE='as_is', SEED=7
SEED     = 7                     # model init + window sampling; the holdout does NOT depend on it (HOLDOUT_SEED)
HOLDOUT_SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)
SNAPSHOT_EPOCHS = (2, 3, 4, 5)   # also write test submissions from the weights of these epochs (the LB-best grid
                                 # weights so far came from epoch 4; the holdout cannot tell which epoch is best)

# ---- data / holdout ----
HOLDOUT_SHIFT  = 1500      # held-out grid sites most shifted toward the test climate
HOLDOUT_RANDOM = 1000      # held-out random grid sites (in-distribution reference)
VAL_KEY        = 'shift'   # holdout that drives best epoch, LR plateau and early stopping: 'shift' | 'random' | 'all'
SUS_FEATS = [35, 39, 43, 99, 103, 107, 111, 115, 119, 123, 127]   # OOD audit: in test their floor sits below the
                                                                   # train minimum -> left out of the shift direction
CO2_FEATS  = list(range(8, 32))      # CO2-like block: smooth +3 sd trend over 1980-2020 in the grid; in the test
                                     # nearly flat, ~no seasonal cycle and a lower level
CLIM_FEATS = list(range(36, 129, 4)) # in the test: one monthly climatology repeated for all 40 years (zero
                                     # interannual variability, zero trend)
HARMONIZE  = False                   # True: give training/holdout/test forcing the test's format (CLIM_FEATS ->
                                     # per-site monthly climatology, CO2_FEATS -> annual mean). Off: the val-shift
                                     # tests showed neither format difference moves LSTM-XL (msd 0.0004 / 0.0012);
                                     # only the CO2 LEVEL does, and harmonizing cannot fix that
SHIFT_EXCLUDE = sorted(set(SUS_FEATS) | {7} | set(CO2_FEATS))   # left out of the shift direction (format
                                     # artifacts / near-uniform CO2 that would otherwise dominate it)
TEST_VARIANTS = ('none',) if CO2_MODE == 'const' else ('co2_hist', 'co2x2')
# 'co2x<f>' : CO2 block = test + f * (grid 1980-2020 series - test); co2x1 == co2_hist. The LB optimum along the
#             global -> global_co2_hist line was f ~ 1.9 (output space), so co2x2 is the input-space guess.
# 'none'    : test forcing as given (after HARMONIZE)
# 'clip'    : + clipped to the grid's per-feature [min, max] (fixes the SUS floor offset)
# 'co2_hist': + CO2_FEATS replaced by the grid's mean 1980-2020 CO2 series (LB: 0.134 -> 0.122 for grid run 1)
# the first variant becomes submission.csv, the others submission_<v>.csv
LOAD_CHUNK = 1024          # sites per read while loading glob_X_fea.npy

# ---- model ----
PRESET   = 'full'          # 'full' = sizes of the LB-0.168 LSTM-XL / LB-0.169 EcoFormer runs | 'lite' = fusion-lite
BACKBONE = 'lstm_xl'       # 'lstm_xl' | 'ecoformer' | 'fusion'
PRESETS = {
    'lite': dict(lstm_kw=dict(d_model=192, n_shared=2, branch_units=160, mlp_units=256, dropout=0.1),
                 eco_kw=dict(d_model=256, n_heads=8, d_ff=768, n_enc=4, n_dec=2, dropout=0.1)),
    'full': dict(lstm_kw=dict(d_model=320, n_shared=3, branch_units=320, mlp_units=512, dropout=0.1),
                 eco_kw=dict(d_model=384, n_heads=8, d_ff=1536, n_enc=6, n_dec=4, dropout=0.1)),
}

# ---- hardware ----
USE_MULTI_GPU = True       # MirroredStrategy over all visible GPUs (falls back to 1 device automatically)
CROSS_DEVICE  = 'nccl'     # 'nccl' | 'hierarchical' (switch if the first step hangs or NCCL errors)
UNROLL_LOOP   = False      # False: K-year rollout as tf.while_loop (compiled once)

# ---- optimisation (batch sizes are GLOBAL = summed over GPUs) ----
BATCH_SIZE   = 1024        # global batch for K=1
ROLL_TOKENS  = 4096        # cap on global batch * K (lower it if K>=4 runs out of GPU memory)
MIN_BATCH    = 64
BASE_LR      = 3e-4        # 5e-4 (LSTM-XL recipe) diverged on the grid: train loss rose during K=1, NaN at K=4
MIN_LR       = 1e-5
WARMUP_STEPS = 1000
CLIP_NORM    = 1.0         # applied to the all-reduced (global) gradient
WEIGHT_DECAY = 1e-4        # LSTM-XL recipe (fusion-lite used 1e-2)
Y_W          = [2., 2., 1., 1., 1., 1., 1.]
CLAMP_NONNEG = True
CLAMP_STE    = True        # straight-through clamp: a prediction below the floor still receives a gradient
Z_CLIP       = 10.0        # standardized forcing clipped to +/- Z_CLIP (grid max |z| is ~36) in training and test
STATE_CEIL   = 2.0         # rollout states capped at STATE_CEIL x the grid's per-variable maximum, so fed-back
                           # predictions cannot run away during K-year training rollouts or inference
NAN_RETRIES  = 3           # a diverged epoch is redone from best.npz with half the LR, at most this many times
DIVERGE_FACTOR = 3.0       # diverged = non-finite steps/weights, or holdout metric > DIVERGE_FACTOR x best so far

# ---- epochs (counted across ALL sessions) / early stop / LR plateau ----
MAX_EPOCHS        = 6       # K=1 only; LB-best grid weights so far: epoch 4
SAMPLES_PER_EPOCH = 2_000_000    # model-steps between evaluations (~6% of one pass over the grid's pairs)
LR_FACTOR, LR_PATIENCE = 0.5, 2
ES_PATIENCE       = 4
MIN_DELTA         = 1e-4

# ---- scheduled-sampling curriculum: (start_epoch, K years/window, teacher-forcing prob) ----
CURRICULUM = [(1, 1, 1.00)]
# K=1 only. On the grid every K=4 stage blew up (run 1: epoch 5, run 2: epoch 9, both at teacher forcing 0.75), and the
# LB-best grid weights are K=1 weights. Scheduled sampling (re-enable at your own risk):
# CURRICULUM = [(1, 1, 1.00), (9, 4, 0.75), (12, 8, 0.50), (15, 12, 0.25), (18, 16, 0.0)]

USE_EMA   = True
EMA_DECAY = 0.998          # ~500-step window; an epoch is ~500 (K=16) to ~2000 (K=1) steps

# ---- sessions / resume ----
SESSION_LIMIT_H   = 11.0   # Kaggle stops a session at 12 h; stay below it
FINAL_RESERVE_MIN = 20 + 6 * 2 * len(TEST_VARIANTS) * (1 + len(SNAPSHOT_EPOCHS))   # ~6 min per test rollout
RESUME            = True
RESUME_FROM       = ''     # optional explicit path to a {RUN_NAME}_ckpt folder

assert CO2_MODE in ('as_is', 'const'), CO2_MODE
assert all(v in ('none', 'clip', 'co2_hist') or v.startswith('co2x') for v in TEST_VARIANTS), TEST_VARIANTS
assert CO2_MODE == 'as_is' or TEST_VARIANTS == ('none',), 'CO2 test variants mean nothing with CO2_MODE=const'

def make_model():
    p = PRESETS[PRESET]
    if BACKBONE == 'lstm_xl':
        return LSTMXLBackbone(**p['lstm_kw'])
    if BACKBONE == 'ecoformer':
        return EcoFormerBackbone(**p['eco_kw'])
    return FusionED(lstm_kw=p['lstm_kw'], eco_kw=p['eco_kw'])

def make_optimizer(lr):
    opt = tf.keras.optimizers.AdamW(learning_rate=lr, weight_decay=WEIGHT_DECAY)
    try:
        opt.exclude_from_weight_decay(var_names=['bias', 'gamma', 'beta', 'layer_normalization',
                                                 'embeddings', 'fuse_gate'])
    except Exception as e:
        print('exclude_from_weight_decay note:', e)
    return opt

print(f'{RUN_NAME}: CO2_MODE={CO2_MODE} SEED={SEED} snapshots {SNAPSHOT_EPOCHS} | PRESET={PRESET} BACKBONE={BACKBONE} | holdout shift {HOLDOUT_SHIFT} + random {HOLDOUT_RANDOM} '
      f'(selection on {VAL_KEY!r}) | curriculum {CURRICULUM}')

global_noco2_s7: CO2_MODE=const SEED=7 snapshots (2, 3, 4, 5) | PRESET=full BACKBONE=lstm_xl | holdout shift 1500 + random 1000 (selection on 'shift') | curriculum [(1, 1, 1.0)]


## 3. Load the global grid

`glob_X_fea.npy` (54,152 × 40 × 12 × 136, float32, 14 GB) is read in chunks, standardized with the same
`X_MEAN/X_STD` as before and stored as float16. `glob_Y{age}.npy` (54,152 × 41 × 12 × 7 per age) keeps month 11,
the annual state, as in every earlier notebook. The `train/val/test.csv` files hold grid indices with no header row.

In [5]:
t0 = time.time()
D = f'{BASE}/data_global'

def read_idx(name):                       # the split files have no header row
    return pd.read_csv(f'{BASE}/{name}.csv', header=None).iloc[:, 0].to_numpy(np.int64)

gx = np.load(f'{D}/glob_X_fea.npy', mmap_mode='r')
N_GLOB = gx.shape[0]
assert gx.shape[1:] == (N_YEAR, N_T, N_CONS), gx.shape
SPLIT = {k: read_idx(k) for k in ('train', 'val', 'test')}
_cat = np.concatenate(list(SPLIT.values()))
print('grid sites:', N_GLOB, '| split files:', {k: len(v) for k, v in SPLIT.items()},
      '| exact partition of the grid:', len(_cat) == N_GLOB and np.array_equal(np.sort(_cat), np.arange(N_GLOB)))

def harmonize(c):
    """In place, raw forcing (..., 40, 12, 136): give it the test's format."""
    if HARMONIZE:
        c[..., CLIM_FEATS] = c[..., CLIM_FEATS].mean(axis=-3, keepdims=True)   # monthly climatology, every year
        c[..., CO2_FEATS]  = c[..., CO2_FEATS].mean(axis=-2, keepdims=True)    # annual mean, no seasonal cycle
    return c

# ---- forcing: harmonized, standardized, stored as float16 ----
x_all  = np.empty((N_GLOB, N_YEAR, N_T, N_CONS), np.float16)
FP_ALL = np.empty((N_GLOB, N_CONS), np.float32)      # per-site 40-year mean of the standardized forcing
XMIN_G = np.full(N_CONS, np.inf, np.float32)         # raw per-feature range over the grid (for 'clip')
XMAX_G = np.full(N_CONS, -np.inf, np.float32)
X_BAD  = np.zeros(N_GLOB, bool)
zmax = 0.0
co2_sum = np.zeros((N_YEAR, N_T, len(CO2_FEATS)))
for s in tqdm(range(0, N_GLOB, LOAD_CHUNK), desc='glob_X'):
    e = min(s + LOAD_CHUNK, N_GLOB)
    c = np.array(gx[s:e], dtype=np.float32)
    bad = ~np.isfinite(c)
    if bad.any():
        X_BAD[s:e] |= bad.any(axis=(1, 2, 3))
        c[bad] = np.broadcast_to(X_MEAN, c.shape)[bad]
    harmonize(c)
    co2_sum += c[..., CO2_FEATS].sum(axis=0)
    XMIN_G = np.minimum(XMIN_G, c.min(axis=(0, 1, 2)))
    XMAX_G = np.maximum(XMAX_G, c.max(axis=(0, 1, 2)))
    c -= X_MEAN; c /= (X_STD + 1e-10)
    zmax = max(zmax, float(np.abs(c).max()))
    FP_ALL[s:e] = c.mean(axis=(1, 2))                 # before clipping: keeps the holdout of earlier runs
    if CO2_MODE == 'const':
        c[..., CO2_FEATS] = 0.0                      # grid mean of the CO2 block
    np.clip(c, -Z_CLIP, Z_CLIP, out=c)
    x_all[s:e] = c
    del c
del gx; gc.collect()
CO2_SERIES = (co2_sum / N_GLOB).astype(np.float32)    # grid-mean CO2 block, (40, 12, 24), raw units
assert zmax < 6e4, f'standardized forcing reaches |z|={zmax:.0f}, too large for float16'
print(f'glob_X -> float16 in {(time.time() - t0) / 60:.1f} min | max |z| {zmax:.1f} (clipped to {Z_CLIP}) | '
      f'sites with non-finite forcing: {int(X_BAD.sum())} | HARMONIZE={HARMONIZE}')

# ---- annual states (month 11) for the 15 seed ages; physical units until the holdout is extracted ----
t1 = time.time()
y_all = np.empty((N_AGES, N_GLOB, N_YEAR + 1, N_OUT), np.float32)
for ai, age in enumerate(tqdm(SEED_AGES, desc='glob_Y')):
    g = np.load(f'{D}/glob_Y{age:03d}.npy', mmap_mode='r')
    assert g.shape == (N_GLOB, N_YEAR + 1, N_T, N_OUT), g.shape
    for s in range(0, N_GLOB, 8192):
        y_all[ai, s:s + 8192] = g[s:s + 8192, :, 11, :]
    del g
SITE_OK = ~X_BAD & np.isfinite(y_all).all(axis=(0, 2, 3))
print(f'glob_Y loaded in {(time.time() - t1) / 60:.1f} min | usable sites {int(SITE_OK.sum())}/{N_GLOB} | '
      f'RAM: x {x_all.nbytes / 1e9:.1f} GB, y {y_all.nbytes / 1e9:.1f} GB')

grid sites: 54152 | split files: {'train': 3035, 'val': 338, 'test': 50779} | exact partition of the grid: True


glob_X:   0%|          | 0/53 [00:00<?, ?it/s]

glob_X -> float16 in 3.4 min | max |z| 36.5 (clipped to 10.0) | sites with non-finite forcing: 0 | HARMONIZE=False


glob_Y:   0%|          | 0/15 [00:00<?, ?it/s]

glob_Y loaded in 1.6 min | usable sites 54152/54152 | RAM: x 7.1 GB, y 0.9 GB


## 4. Holdout sites

The OOD audit showed the test sites sit far from every training site in climate space (median nearest-site
distance ~3.5 vs ~1.0 for the old val set), so a random holdout says little about the test. Two holdouts are kept
out of training:

* **shift**: the mean difference between the test and grid climate fingerprints (40-year mean of each standardized
  feature, `SHIFT_EXCLUDE` left out: the SUS floor artifact, feature 7 and the
  near-uniform CO2 block, which would otherwise dominate it) defines a shift direction; the `HOLDOUT_SHIFT` grid sites projecting furthest along
  it are held out. The model must extrapolate toward test-like climate to score well on them.
* **random**: `HOLDOUT_RANDOM` random grid sites, the in-distribution reference.

Sites the earlier runs trained on (`res_train4_test8` `x_train`, matched in the grid by fingerprint) are never put in a
holdout, so old models can later be scored on the same sites (`global_lstmxl_holdout.npz`).

In [6]:
def nearest_rows(a, b, chunk=512):
    """For every row of a: index of the nearest row of b and the Euclidean distance."""
    a = np.asarray(a, np.float64); b = np.asarray(b, np.float64)
    bb = (b ** 2).sum(1)
    j = np.empty(len(a), np.int64); d = np.empty(len(a))
    for s in range(0, len(a), chunk):
        d2 = (a[s:s + chunk] ** 2).sum(1)[:, None] + bb[None] - 2.0 * a[s:s + chunk] @ b.T
        j[s:s + chunk] = d2.argmin(1)
        d[s:s + chunk] = np.sqrt(np.maximum(d2.min(1), 0.0))
    return j, d

# ---- sites the earlier runs trained on -> never part of a holdout ----
_res = np.load(f'{BASE}/data_global/res_train4_test8.npz')
fp_old = normData(_res['x_train'].mean(axis=(1, 2)), X_MEAN, X_STD)
del _res; gc.collect()
j_old, d_old = nearest_rows(fp_old, FP_ALL)
_csv_old = np.union1d(SPLIT['train'], SPLIT['val'])
OLD_TRAIN = np.unique(j_old[d_old < 1e-3])
print(f'earlier-run train sites found in the grid: {len(OLD_TRAIN)}/{len(fp_old)} '
      f'| same set as train.csv + val.csv: {np.array_equal(OLD_TRAIN, _csv_old)}')
OLD_TRAIN = np.union1d(OLD_TRAIN, _csv_old)

# ---- shift direction: mean test fingerprint - mean grid fingerprint ----
KEEP_F = np.setdiff1d(np.arange(N_CONS), SHIFT_EXCLUDE)
fp_test = []
for scn in SCENARIOS:
    _tx = np.load(f'{COMP}/test_{scn}.npz')['test_x']
    fp_test.append(normData(_tx.mean(axis=(1, 2)), X_MEAN, X_STD))
    print(f'{scn}: CO2 block level, test - grid (40-year mean, in X_STD units): '
          f'{float(np.mean(fp_test[-1][:, CO2_FEATS]) - np.mean(FP_ALL[:, CO2_FEATS])):+.2f}')
    del _tx; gc.collect()
fp_test = np.concatenate(fp_test).astype(np.float32)
mu_g   = FP_ALL[:, KEEP_F].mean(0)
shift  = fp_test[:, KEEP_F].mean(0) - mu_g
u      = shift / (np.linalg.norm(shift) + 1e-12)
proj_g = (FP_ALL[:, KEEP_F] - mu_g) @ u
proj_t = (fp_test[:, KEEP_F] - mu_g) @ u

cand = np.setdiff1d(np.flatnonzero(SITE_OK), OLD_TRAIN)
hold_shift = np.sort(cand[np.argsort(-proj_g[cand])[:HOLDOUT_SHIFT]])
rest = np.setdiff1d(cand, hold_shift)
hold_rand = np.sort(np.random.default_rng(HOLDOUT_SEED).choice(rest, size=min(HOLDOUT_RANDOM, len(rest)), replace=False))
HOLD_SITES = np.concatenate([hold_shift, hold_rand]).astype(np.int64)
HOLD_GROUPS = {k: v for k, v in (('shift', np.arange(len(hold_shift))),
                                 ('random', np.arange(len(hold_shift), len(HOLD_SITES))),
                                 ('all', np.arange(len(HOLD_SITES)))) if len(v)}
assert VAL_KEY in HOLD_GROUPS, f'VAL_KEY={VAL_KEY!r} needs a non-empty holdout'
TRAIN_SITES = np.setdiff1d(np.flatnonzero(SITE_OK), HOLD_SITES)
N_TRAIN_SITES = len(TRAIN_SITES)
HOLD_HASH = int(zlib.crc32(HOLD_SITES.tobytes()))

thr = float(proj_g[hold_shift].min()) if len(hold_shift) else np.inf
print(f'shift direction: |mean(test) - mean(grid)| = {np.linalg.norm(shift):.2f} z over {len(KEEP_F)} features')
print(f'projection: grid p50 {np.median(proj_g):+.2f} p99 {np.percentile(proj_g, 99):+.2f} | '
      f'shift holdout >= {thr:+.2f} | test p50 {np.median(proj_t):+.2f}, '
      f'{np.mean(proj_t >= thr):.1%} of test sites beyond the holdout threshold')
print(f'train sites {N_TRAIN_SITES} | holdout shift {len(hold_shift)} + random {len(hold_rand)} | hash {HOLD_HASH}')
np.savez(f'{OUT}/{RUN_NAME}_holdout.npz', shift=hold_shift, random=hold_rand, old_train=OLD_TRAIN)

# ---- holdout truth (physical), then standardize the states in place ----
hold_y0  = y_all[:, HOLD_SITES, 0, :].copy()          # (15, Nh, 7)
hold_y40 = y_all[:, HOLD_SITES, N_YEAR, :].copy()
PHYS_MAX = y_all.max(axis=(0, 1, 2))                  # per-variable maximum over the grid, physical units
CEIL_N = normData(STATE_CEIL * PHYS_MAX, Y_MEAN, Y_STD).astype(np.float32)
y_all -= Y_MEAN; y_all /= (Y_STD + 1e-10)
y_n = y_all
print('state ceiling (physical):', np.round(STATE_CEIL * PHYS_MAX, 3))
print(f'one-step pairs available for training: {N_AGES * N_TRAIN_SITES * N_YEAR:,} '
      f'(earlier runs: {N_AGES * 3373 * N_YEAR:,})')

earlier-run train sites found in the grid: 3373/3373 | same set as train.csv + val.csv: True
ssp126: CO2 block level, test - grid (40-year mean, in X_STD units): -0.75
ssp585: CO2 block level, test - grid (40-year mean, in X_STD units): -0.38
shift direction: |mean(test) - mean(grid)| = 0.94 z over 100 features
projection: grid p50 +0.43 p99 +3.71 | shift holdout >= +3.10 | test p50 +1.34, 13.0% of test sites beyond the holdout threshold
train sites 51652 | holdout shift 1500 + random 1000 | hash 1869871560
state ceiling (physical): [ 70.     72.322 213.836  23.221  14.709   7.307  14.592]
one-step pairs available for training: 30,991,200 (earlier runs: 2,023,800)


## 5. Models — LSTM-XL, EcoFormer, FusionED (identical code to fusion-lite / fusion-v2)

In [7]:
# ---------------- LSTM-XL ----------------
class GatedBranch(layers.Layer):
    def __init__(self, idx, units, mlp_units, dropout, **kw):
        super().__init__(name=f'branch{idx}', **kw)
        self.idx = idx
        self.lstm_data = layers.LSTM(units, return_sequences=True)
        self.lstm_gate = layers.LSTM(units, return_sequences=True)
        self.gate      = layers.Dense(units, activation='sigmoid')
        self.lstm_out  = layers.LSTM(units, return_sequences=True)
        self.norm      = layers.LayerNormalization(epsilon=1e-5)
        self.pool      = layers.Dense(1)
        self.mlp1      = layers.Dense(mlp_units, activation='gelu')
        self.mlp2      = layers.Dense(128, activation='gelu')
        self.drop      = layers.Dropout(dropout)
        self.head      = layers.Dense(1, kernel_initializer='zeros', bias_initializer='zeros')
    def call(self, h, prev, age, training=False):
        i = self.idx; T = tf.shape(h)[1]
        age_i = tf.concat([age[:, :1], age[:, i+1:i+2], age[:, i+8:i+9]], axis=-1)
        loc = tf.concat([prev[:, i:i+1], age_i], axis=-1)
        z = tf.concat([h, tf.repeat(loc[:, None, :], T, axis=1)], axis=-1)
        s = self.lstm_data(z) * self.gate(self.lstm_gate(z))
        s = self.norm(self.lstm_out(s))
        w = tf.nn.softmax(self.pool(s), axis=1)
        feat = tf.concat([tf.reduce_sum(w * s, axis=1), s[:, -1]], axis=-1)
        feat = self.mlp2(self.drop(self.mlp1(feat), training=training))
        feat = tf.concat([feat, prev, age_i], axis=-1)
        return prev[:, i:i+1] + self.head(feat)

class LSTMXLBackbone(keras.Model):
    """Returns the DELTA (B,7) so it can be fused; add prev outside for standalone use."""
    def __init__(self, n_out=N_OUT, n_cons=N_CONS, n_age_fea=N_AGE_FEA, d_model=320, n_shared=3,
                 branch_units=320, mlp_units=512, dropout=0.1, return_delta=False, **kw):
        super().__init__(**kw)
        self.n_out, self.n_cons, self.n_age_fea, self.return_delta = n_out, n_cons, n_age_fea, return_delta
        self.ctx_proj = layers.Dense(128, activation='gelu')
        self.in_proj  = layers.Dense(d_model)
        self.in_norm  = layers.LayerNormalization(epsilon=1e-5)
        self.enc_lstms = [layers.Bidirectional(layers.LSTM(d_model // 2, return_sequences=True)) for _ in range(n_shared)]
        self.enc_norms = [layers.LayerNormalization(epsilon=1e-5) for _ in range(n_shared)]
        self.enc_drop  = layers.Dropout(dropout)
        self.branches  = [GatedBranch(i, branch_units, mlp_units, dropout) for i in range(n_out)]
    def call(self, inputs, training=False):
        nc, no, na = self.n_cons, self.n_out, self.n_age_fea
        env = inputs[:, :, :nc]; prev = inputs[:, -1, nc:nc+no]; age = inputs[:, -1, nc+no:nc+no+na]
        T = tf.shape(env)[1]
        ctx = self.ctx_proj(tf.concat([prev, age], axis=-1))
        h = tf.concat([env, tf.repeat(ctx[:, None, :], T, axis=1)], axis=-1)
        h = self.in_norm(self.in_proj(h))
        for lstm, norm in zip(self.enc_lstms, self.enc_norms):
            h = norm(h + self.enc_drop(lstm(h), training=training))
        out = tf.concat([b(h, prev, age, training=training) for b in self.branches], axis=-1)  # prev+delta
        return (out - prev) if self.return_delta else out

# ---------------- EcoFormer ----------------
class EcoEncBlock(layers.Layer):
    def __init__(self, d, heads, d_ff, drop, **kw):
        super().__init__(**kw)
        self.n1 = layers.LayerNormalization(epsilon=1e-5)
        self.att = layers.MultiHeadAttention(heads, d // heads, dropout=drop)
        self.n2 = layers.LayerNormalization(epsilon=1e-5)
        self.f1 = layers.Dense(d_ff, activation='gelu'); self.f2 = layers.Dense(d)
        self.drop = layers.Dropout(drop)
    def call(self, x, training=False):
        y = self.n1(x); x = x + self.drop(self.att(y, y, training=training), training=training)
        y = self.n2(x); return x + self.drop(self.f2(self.f1(y)), training=training)

class EcoDecBlock(layers.Layer):
    def __init__(self, d, heads, d_ff, drop, **kw):
        super().__init__(**kw)
        self.n1 = layers.LayerNormalization(epsilon=1e-5)
        self.self_att = layers.MultiHeadAttention(heads, d // heads, dropout=drop)
        self.n2 = layers.LayerNormalization(epsilon=1e-5)
        self.cross_att = layers.MultiHeadAttention(heads, d // heads, dropout=drop)
        self.n3 = layers.LayerNormalization(epsilon=1e-5)
        self.f1 = layers.Dense(d_ff, activation='gelu'); self.f2 = layers.Dense(d)
        self.drop = layers.Dropout(drop)
    def call(self, q, mem, training=False):
        y = self.n1(q); q = q + self.drop(self.self_att(y, y, training=training), training=training)
        y = self.n2(q); q = q + self.drop(self.cross_att(y, mem, training=training), training=training)
        y = self.n3(q); return q + self.drop(self.f2(self.f1(y)), training=training)

class EcoFormerBackbone(keras.Model):
    def __init__(self, n_out=N_OUT, n_cons=N_CONS, n_age_fea=N_AGE_FEA, n_months=N_T, d_model=384,
                 n_heads=8, d_ff=1536, n_enc=6, n_dec=4, dropout=0.1, return_delta=False, **kw):
        super().__init__(**kw)
        self.n_out, self.n_cons, self.n_age_fea, self.n_months = n_out, n_cons, n_age_fea, n_months
        self.return_delta = return_delta
        self.env_proj  = layers.Dense(d_model); self.env_norm = layers.LayerNormalization(epsilon=1e-5)
        self.month_emb = layers.Embedding(n_months, d_model)
        self.state_in1 = layers.Dense(d_model, activation='gelu'); self.state_in2 = layers.Dense(d_model)
        self.enc = [EcoEncBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_enc)]
        self.enc_norm = layers.LayerNormalization(epsilon=1e-5)
        self.var_emb = layers.Embedding(n_out, d_model)
        self.var_in1 = layers.Dense(d_model, activation='gelu'); self.var_in2 = layers.Dense(d_model)
        self.dec = [EcoDecBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_dec)]
        self.dec_norm = layers.LayerNormalization(epsilon=1e-5)
        self.head1 = layers.Dense(d_model, activation='gelu')
        self.head2 = layers.Dense(1, kernel_initializer='zeros', bias_initializer='zeros')
    def call(self, inputs, training=False):
        nc, no, na = self.n_cons, self.n_out, self.n_age_fea
        env = inputs[:, :, :nc]; prev = inputs[:, -1, nc:nc+no]; age = inputs[:, -1, nc+no:nc+no+na]
        h = self.env_norm(self.env_proj(env)) + self.month_emb(tf.range(self.n_months))[None]
        s = self.state_in2(self.state_in1(tf.concat([prev, age], axis=-1)))[:, None, :]
        h = tf.concat([s, h], axis=1)
        for blk in self.enc: h = blk(h, training=training)
        mem = self.enc_norm(h)
        age_tri = tf.stack([tf.stack([age[:, 0], age[:, 1+i], age[:, 8+i]], axis=-1) for i in range(no)], axis=1)
        vf = tf.concat([prev[..., None], age_tri], axis=-1)
        q = self.var_in2(self.var_in1(vf)) + self.var_emb(tf.range(no))[None]
        for blk in self.dec: q = blk(q, mem, training=training)
        delta = self.head2(self.head1(self.dec_norm(q)))[..., 0]
        return delta if self.return_delta else (prev + delta)

# ---------------- Fusion ----------------
class FusionED(keras.Model):
    """Run both backbones, blend their per-variable deltas with a learned gate, residual-add."""
    def __init__(self, lstm_kw=None, eco_kw=None, **kw):
        super().__init__(**kw)
        self.lstm = LSTMXLBackbone(return_delta=True, **(lstm_kw or {}))
        self.eco  = EcoFormerBackbone(return_delta=True, **(eco_kw or {}))
        # per-variable gate in [0,1]; init 0 -> sigmoid 0.5 (equal blend)
        self.gate = tf.Variable(tf.zeros((N_OUT,)), trainable=True, name='fuse_gate')
    def call(self, inputs, training=False):
        prev = inputs[:, -1, N_CONS:N_CONS+N_OUT]
        d_lstm = self.lstm(inputs, training=training)
        d_eco  = self.eco(inputs, training=training)
        g = tf.sigmoid(self.gate)[None, :]
        return prev + g * d_lstm + (1.0 - g) * d_eco

print('models defined: LSTMXLBackbone, EcoFormerBackbone, FusionED')

models defined: LSTMXLBackbone, EcoFormerBackbone, FusionED


## 6. Distribution strategy & model build

In [8]:
gpus = tf.config.list_physical_devices('GPU')
if USE_MULTI_GPU and len(gpus) > 1:
    _cdo = (tf.distribute.NcclAllReduce() if CROSS_DEVICE == 'nccl'
            else tf.distribute.HierarchicalCopyAllReduce())
    STRATEGY = tf.distribute.MirroredStrategy(cross_device_ops=_cdo)
else:
    STRATEGY = tf.distribute.get_strategy()      # single GPU / CPU
NREP = STRATEGY.num_replicas_in_sync
print('GPUs visible:', len(gpus), '| replicas in sync:', NREP, '|', type(STRATEGY).__name__)

GPUs visible: 1 | replicas in sync: 1 | _DefaultDistributionStrategy


In [9]:
with STRATEGY.scope():
    MODEL = make_model()
    _ = MODEL(tf.zeros((2, N_T, N_FEA), tf.float32), training=False)     # build all variables
    OPT = make_optimizer(BASE_LR)
    OPT.build(MODEL.trainable_variables)
    TVARS = list(MODEL.trainable_variables)
    EMA = ([tf.Variable(v.numpy(), trainable=False, name=f'ema_{i}') for i, v in enumerate(TVARS)]
           if USE_EMA else [])
print(f'{RUN_NAME}/{BACKBONE}/{PRESET}: {MODEL.count_params() / 1e6:.2f}M params')

I0000 00:00:1791334655.024104      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5


global_noco2_s7/lstm_xl/full: 22.72M params


## 7. Training engine

Same engine as fusion-lite: window sampler → `tf.data` → replicas, K-year rollout loss with scheduled sampling,
all-reduced and globally clipped gradients, AdamW, EMA. Changes: windows are drawn from the grid's training sites
(float16 forcing cast per batch), the clamp is straight-through when `CLAMP_STE`, and the rollout takes a per-year
forcing function so the holdout and the test variants share one code path.

In [10]:
FLOOR_T = tf.constant(FLOOR_N)
CEIL_T  = tf.constant(CEIL_N)
Y_W_T   = tf.constant(Y_W, tf.float32)

# ---------------------------------------------------------------- data
def global_batch_for(K):
    b = int(min(BATCH_SIZE, max(MIN_BATCH, ROLL_TOKENS // K)))
    return max(NREP, b - b % NREP)                   # divisible by the number of replicas

def window_batches(K, n_windows, batch, rng):
    """Yield standardized (env (B,K,12,136), states (B,K+1,7), age_feat (B,15)) from random training windows."""
    T0 = N_YEAR - K + 1
    total = N_AGES * N_TRAIN_SITES * T0              # ~31M windows: repeats inside an epoch are negligible
    kx, ky = np.arange(K), np.arange(K + 1)
    for _ in range(n_windows // batch):
        b = rng.integers(0, total, size=batch)
        ai, rem = np.divmod(b, N_TRAIN_SITES * T0)
        pos, t0 = np.divmod(rem, T0)
        site = TRAIN_SITES[pos]
        env = x_all[site[:, None], t0[:, None] + kx].astype(np.float32)
        yy  = y_n[ai[:, None], site[:, None], t0[:, None] + ky]
        yield env, yy, AGE_FEAT[ai]

def make_dataset(K, n_windows, batch, rng):
    sig = (tf.TensorSpec((batch, K, N_T, N_CONS), tf.float32),
           tf.TensorSpec((batch, K + 1, N_OUT), tf.float32),
           tf.TensorSpec((batch, N_AGE_FEA), tf.float32))
    ds = tf.data.Dataset.from_generator(lambda: window_batches(K, n_windows, batch, rng),
                                        output_signature=sig).prefetch(8)
    opts = tf.data.Options()
    opts.experimental_distribute.auto_shard_policy = tf.data.experimental.AutoShardPolicy.OFF
    return STRATEGY.experimental_distribute_dataset(ds.with_options(opts))

# ---------------------------------------------------------------- rollout loss (per replica)
def _clamp_train(pred):
    lo = FLOOR_T if CLAMP_NONNEG else tf.fill(tf.shape(CEIL_T), -1e9)
    c = tf.clip_by_value(pred, lo, CEIL_T)
    return pred + tf.stop_gradient(c - pred) if CLAMP_STE else c     # STE: forward = clamp, backward = identity

def _predict(env_k, prev, af_m):
    prev_m = tf.repeat(prev[:, None, :], N_T, axis=1)
    return _clamp_train(MODEL(tf.concat([env_k, prev_m, af_m], axis=-1), training=True))

def rollout_loss_unrolled(env, yy, af, tf_ratio):
    """Original engine: python loop over a static K."""
    K = env.shape[1]; b = tf.shape(env)[0]
    af_m = tf.repeat(af[:, None, :], N_T, axis=1)
    prev, loss = yy[:, 0], 0.0
    for k in range(K):
        pred = _predict(env[:, k], prev, af_m)
        loss += tf.reduce_mean(Y_W_T * tf.square(pred - yy[:, k + 1]))
        if k < K - 1:
            keep = tf.cast(tf.random.uniform((b, 1)) < tf_ratio, tf.float32)
            prev = keep * yy[:, k + 1] + (1.0 - keep) * pred
    return loss / float(K)

def rollout_loss_while(env, yy, af, tf_ratio):
    """Same maths as rollout_loss_unrolled, but the model is traced once for any K."""
    K = tf.shape(env)[1]; b = tf.shape(env)[0]
    af_m = tf.repeat(af[:, None, :], N_T, axis=1)
    def body(k, prev, loss):
        tgt  = tf.gather(yy, k + 1, axis=1)
        pred = _predict(tf.gather(env, k, axis=1), prev, af_m)
        loss = loss + tf.reduce_mean(Y_W_T * tf.square(pred - tgt))
        keep = tf.cast(tf.random.uniform((b, 1)) < tf_ratio, tf.float32)
        return k + 1, keep * tgt + (1.0 - keep) * pred, loss
    _, _, loss = tf.while_loop(lambda k, p, l: k < K, body,
                               (tf.constant(0), yy[:, 0], tf.constant(0.0)))
    return loss / tf.cast(K, tf.float32)

ROLLOUT_LOSS = rollout_loss_unrolled if UNROLL_LOOP else rollout_loss_while

# ---------------------------------------------------------------- distributed step
def replica_step(env, yy, af, tf_ratio):
    with tf.GradientTape() as tape:
        loss = ROLLOUT_LOSS(env, yy, af, tf_ratio) / NREP      # local mean / replicas
    grads = tape.gradient(loss, TVARS)
    grads = [tf.zeros_like(v) if g is None else tf.convert_to_tensor(g) for g, v in zip(grads, TVARS)]
    if NREP > 1:
        grads = tf.distribute.get_replica_context().all_reduce(tf.distribute.ReduceOp.SUM, grads)
    gnorm = tf.linalg.global_norm(grads)                           # norm of the GLOBAL gradient (same on every replica)
    ok = tf.math.is_finite(gnorm)                                  # NaN/inf anywhere -> skip this update
    grads = [tf.where(ok, g, tf.zeros_like(g)) for g in grads]
    grads, _ = tf.clip_by_global_norm(grads, CLIP_NORM, use_norm=tf.where(ok, gnorm, 0.0))
    if NREP > 1:
        grads = [g / NREP for g in grads]                          # the optimizer sums replicas again
    OPT.apply_gradients(zip(grads, TVARS))
    return tf.where(ok, loss, 0.0), 1.0 - tf.cast(ok, tf.float32)

def _dist_step(env, yy, af, tf_ratio):
    loss, skipped = STRATEGY.run(replica_step, args=(env, yy, af, tf_ratio))
    loss    = STRATEGY.reduce(tf.distribute.ReduceOp.SUM, loss, axis=None)
    skipped = STRATEGY.reduce(tf.distribute.ReduceOp.SUM, skipped, axis=None)   # same flag on every replica
    for e, v in zip(EMA, TVARS):
        e.assign(EMA_DECAY * e + (1.0 - EMA_DECAY) * v)
    return loss, skipped

dist_step = tf.function(_dist_step, reduce_retracing=not UNROLL_LOOP)

# ---------------------------------------------------------------- inference / metric
@tf.function(reduce_retracing=True)
def infer(x):
    return MODEL(x, training=False)

def rollout(env_at, n_sites, y0_raw, desc=None, bs=8192):
    """40-year autoregressive rollout, all seed ages at once.
    env_at(t) -> standardized float32 forcing of year t, (n_sites,12,136); y0_raw: (15,n_sites,7) physical.
    Returns the year-40 state (15,n_sites,7), physical."""
    N, A = n_sites, y0_raw.shape[0]
    prev = normData(y0_raw, Y_MEAN, Y_STD).reshape(A * N, N_OUT).astype(np.float32)
    site_idx = np.tile(np.arange(N), A)
    af_rows = AGE_FEAT[np.repeat(np.arange(A), N)]
    years = tqdm(range(N_YEAR), desc=desc, leave=False) if desc else range(N_YEAR)
    for t in years:
        env_t = env_at(t)
        nxt = np.empty_like(prev)
        for s in range(0, A * N, bs):
            e = min(s + bs, A * N)
            xb = np.concatenate([env_t[site_idx[s:e]],
                                 np.repeat(prev[s:e, None, :], N_T, axis=1),
                                 np.repeat(af_rows[s:e, None, :], N_T, axis=1)], axis=-1)
            p = infer(tf.constant(xb, tf.float32)).numpy()
            nxt[s:e] = np.minimum(np.maximum(p, FLOOR_N) if CLAMP_NONNEG else p, CEIL_N)
        prev = nxt
    out = invNormData(prev, Y_MEAN, Y_STD).reshape(A, N, N_OUT)
    return np.maximum(out, 0.0) if CLAMP_NONNEG else out

def hold_env(t):
    return x_all[HOLD_SITES, t].astype(np.float32)

def evaluate_holdout(desc=None):
    """Scored metric per holdout group from one rollout over all holdout sites."""
    pred = rollout(hold_env, len(HOLD_SITES), hold_y0, desc=desc)
    return {k: scored_metric(pred[:, idx], hold_y40[:, idx]) for k, idx in HOLD_GROUPS.items()}, pred

def weights_finite():
    return all(np.isfinite(v.numpy()).all() for v in TVARS) and all(np.isfinite(e.numpy()).all() for e in EMA)

def get_tvars():
    return [v.numpy() for v in TVARS]

def set_tvars(ws):
    for v, w in zip(TVARS, ws):
        v.assign(w)

# ---------------------------------------------------------------- LR schedule (state is checkpointed)
def set_lr(v):
    OPT.learning_rate = float(v)

class LRController:
    """Linear warm-up per step, then ReduceLROnPlateau per epoch on the validation metric."""
    def __init__(self):
        self.step, self.best, self.wait = 0, np.inf, 0
        self.lr = BASE_LR * (0.1 if WARMUP_STEPS > 0 else 1.0)
        set_lr(self.lr)
    def on_step(self):
        if self.step < WARMUP_STEPS:
            self.lr = BASE_LR * (0.1 + 0.9 * (self.step + 1) / WARMUP_STEPS)
            set_lr(self.lr)
        self.step += 1
    def on_epoch(self, metric, counting=True):
        if metric < self.best - MIN_DELTA:
            self.best, self.wait = metric, 0
        elif counting and self.step >= WARMUP_STEPS:
            self.wait += 1
            if self.wait >= LR_PATIENCE and self.lr > MIN_LR:
                self.lr = max(self.lr * LR_FACTOR, MIN_LR); set_lr(self.lr); self.wait = 0
                print(f'    ReduceLROnPlateau -> lr = {self.lr:.2e}')
    def state(self):
        return dict(step=self.step, best=None if not np.isfinite(self.best) else float(self.best),
                    wait=self.wait, lr=float(self.lr))
    def load(self, d):
        self.step, self.wait, self.lr = int(d['step']), int(d['wait']), float(d['lr'])
        self.best = np.inf if d['best'] is None else float(d['best'])
        set_lr(self.lr)

def stage_for_epoch(ep):
    si = 0
    for j, (start, _, _) in enumerate(CURRICULUM):
        if ep >= start:
            si = j
    _, K, tfr = CURRICULUM[si]
    return si, K, tfr

print('engine ready | rollout loss:', ROLLOUT_LOSS.__name__, '| STE clamp:', CLAMP_STE, '| NaN-safe steps',
      '| global batch per K:', {K: global_batch_for(K) for _, K, _ in CURRICULUM})

engine ready | rollout loss: rollout_loss_while | STE clamp: True | NaN-safe steps | global batch per K: {1: 1024}


## 8. Checkpoint & resume

Same format as fusion-lite: `<RUN_NAME>_ckpt/` holds `last.npz` (weights, EMA, optimizer moments), `best.npz` (best
weights on the `VAL_KEY` holdout) and `state.json`, written atomically. The fingerprint also covers the holdout, so a
session with different holdout settings cannot silently resume.

In [11]:
CKPT_DIR = os.path.join(OUT, f'{RUN_NAME}_ckpt')
FINGERPRINT = dict(run=RUN_NAME, preset=PRESET, backbone=BACKBONE, n_params=int(MODEL.count_params()),
                   n_tvars=len(TVARS), n_opt_vars=len(OPT.variables),
                   holdout_hash=HOLD_HASH, n_train_sites=int(N_TRAIN_SITES), harmonize=bool(HARMONIZE), co2_mode=CO2_MODE)

def _atomic_savez(path, arrs):
    tmp = path[:-4] + '.tmp.npz'
    np.savez(tmp, **arrs)
    os.replace(tmp, path)

def _atomic_json(path, obj):
    tmp = path + '.tmp'
    with open(tmp, 'w') as f:
        json.dump(obj, f, indent=1)
    os.replace(tmp, path)

def save_ckpt(state, best_weights=None):
    os.makedirs(CKPT_DIR, exist_ok=True)
    arrs = {f'm{i}': w for i, w in enumerate(MODEL.get_weights())}
    arrs.update({f'e{i}': e.numpy() for i, e in enumerate(EMA)})
    arrs.update({f'o{i}': v.numpy() for i, v in enumerate(OPT.variables)})
    _atomic_savez(os.path.join(CKPT_DIR, 'last.npz'), arrs)
    if best_weights is not None:
        _atomic_savez(os.path.join(CKPT_DIR, 'best.npz'), {f'b{i}': w for i, w in enumerate(best_weights)})
    state['lr_ctl'] = lr_ctl.state()
    state['fingerprint'] = FINGERPRINT
    state['saved_at'] = time.strftime('%Y-%m-%d %H:%M:%S')
    _atomic_json(os.path.join(CKPT_DIR, 'state.json'), state)   # written last = commit point

def _read_state(d):
    sj = os.path.join(d, 'state.json')
    if not (os.path.isfile(sj) and os.path.isfile(os.path.join(d, 'last.npz'))):
        return None
    try:
        with open(sj) as f:
            return json.load(f)
    except Exception as e:
        print('  unreadable checkpoint', d, e)
        return None

def find_resume():
    if not RESUME:
        return None, None
    if RESUME_FROM:
        st = _read_state(RESUME_FROM)
        if st is None:
            raise FileNotFoundError(f'RESUME_FROM={RESUME_FROM!r} has no state.json + last.npz')
        return RESUME_FROM, st
    cands = [CKPT_DIR] + [os.path.dirname(p) for p in
                          glob.glob(f'/kaggle/input/**/{RUN_NAME}_ckpt/state.json', recursive=True)]
    best = (None, None)
    for d in cands:
        st = _read_state(d)
        if st is None:
            continue
        key = (st.get('epoch_done', 0), st.get('lr_ctl', {}).get('step', 0))
        if best[1] is None or key > (best[1].get('epoch_done', 0), best[1].get('lr_ctl', {}).get('step', 0)):
            best = (d, st)
    return best

def load_ckpt(d, st):
    fp = st.get('fingerprint', {})
    legacy = dict(co2_mode='as_is')                              # checkpoints written before the key existed
    for k in ('preset', 'backbone', 'n_params', 'n_tvars', 'holdout_hash', 'n_train_sites', 'harmonize', 'co2_mode'):
        if fp.get(k, legacy.get(k)) != FINGERPRINT[k]:
            raise ValueError(f'checkpoint {d} was made with {k}={fp.get(k)!r} but this run has '
                             f'{k}={FINGERPRINT[k]!r}. Use the same PRESET/BACKBONE/holdout settings, or set RESUME=False.')
    if os.path.abspath(d) != os.path.abspath(CKPT_DIR):          # e.g. read-only /kaggle/input copy
        shutil.copytree(d, CKPT_DIR, dirs_exist_ok=True)
    z = np.load(os.path.join(CKPT_DIR, 'last.npz'))
    MODEL.set_weights([z[f'm{i}'] for i in range(len(MODEL.get_weights()))])
    if EMA:
        if 'e0' in z.files:
            for i, e in enumerate(EMA):
                e.assign(z[f'e{i}'])
        else:
            for e, v in zip(EMA, TVARS):
                e.assign(v)
    ov = OPT.variables
    if all(f'o{i}' in z.files and z[f'o{i}'].shape == tuple(ov[i].shape) for i in range(len(ov))):
        for i, v in enumerate(ov):
            v.assign(z[f'o{i}'])
    else:
        print('  optimizer state incompatible -> fresh optimizer moments (weights still restored)')
    if 'lr_ctl' in st:
        lr_ctl.load(st['lr_ctl'])
    if not weights_finite():                                     # e.g. a run that diverged to NaN
        bp = os.path.join(CKPT_DIR, 'best.npz')
        if not os.path.isfile(bp):
            raise ValueError(f'{d}: last.npz is not finite and there is no best.npz -> set RESUME=False')
        zb = np.load(bp)
        set_tvars([zb[f'b{i}'] for i in range(len(TVARS))])
        for e, v in zip(EMA, TVARS):
            e.assign(v)
        for v in OPT.variables:                                  # moments are NaN too -> fresh optimizer state
            v.assign(tf.zeros_like(v))
        be = int(st.get('best_epoch') or 0)
        hist = [h for h in st.get('history', []) if h['epoch'] <= be]
        lr0 = hist[-1]['lr'] if hist else BASE_LR
        st.update(epoch_done=be, history=hist, stage=-1, wait=0, finished=False, finish_kind='')
        lr_ctl.lr = max(0.5 * lr0, MIN_LR); lr_ctl.wait = 0; set_lr(lr_ctl.lr)
        print(f'  last.npz is non-finite -> restarted from best.npz (epoch {be}) with fresh optimizer moments '
              f'and lr {lr_ctl.lr:.1e}')
    print(f'  resumed from {d}: epoch_done={st["epoch_done"]}, best VAL={st.get("best_score")} '
          f'(epoch {st.get("best_epoch")}), optimizer step {int(OPT.iterations.numpy())}, lr {lr_ctl.lr:.2e}')
    return st

## 9. Train (resumes automatically, stops before the session limit)

In [12]:
lr_ctl = LRController()
state = dict(epoch_done=0, best_score=None, best_epoch=0, best_src='', wait=0, stage=-1,
             finished=False, finish_kind='', history=[], sessions=0, last_epoch_sec=0.0, train_hours=0.0,
             nan_retries=0)
src_dir, src_state = find_resume()
if src_dir is not None:
    state.update(load_ckpt(src_dir, src_state))
else:
    print('  no checkpoint found -> training from scratch')
state['sessions'] += 1

# a finished run re-opens if the stop condition no longer holds with this session's config
if state['finished']:
    still = ((state['finish_kind'] == 'early_stop' and state['wait'] >= ES_PATIENCE) or
             (state['finish_kind'] == 'max_epochs' and state['epoch_done'] >= MAX_EPOCHS))
    if not still:
        state['finished'] = False
        print(f'  run was finished ({state["finish_kind"]}) but MAX_EPOCHS={MAX_EPOCHS} / '
              f'ES_PATIENCE={ES_PATIENCE} allow more -> continuing')

DEADLINE = NB_T0 + SESSION_LIMIT_H * 3600 - FINAL_RESERVE_MIN * 60
print(f'session {state["sessions"]}: epochs done {state["epoch_done"]}/{MAX_EPOCHS}, '
      f'{(DEADLINE - time.time()) / 3600:.2f} h of training time left in this session')

stop_reason = 'already finished' if state['finished'] else None

def roll_back(reason):
    """Diverged epoch: restore best.npz (fresh optimizer moments, EMA = weights) with half the LR, so the same epoch
    is redone. Returns False when no retries are left (best weights restored either way)."""
    global stop_reason
    bp = os.path.join(CKPT_DIR, 'best.npz')
    r = state.get('nan_retries', 0)
    if os.path.isfile(bp):
        zb = np.load(bp)
        set_tvars([zb[f'b{k}'] for k in range(len(TVARS))])
        for e, v in zip(EMA, TVARS):
            e.assign(v)
        for v in OPT.variables:
            v.assign(tf.zeros_like(v))
    if r >= NAN_RETRIES or not os.path.isfile(bp):
        stop_reason = f'epoch {ep} diverged ({reason}); no rollback left (retries {r}/{NAN_RETRIES})'
        return False
    state['nan_retries'] = r + 1
    lr_ctl.lr = max(0.5 * lr_ctl.lr, MIN_LR); lr_ctl.wait = 0; set_lr(lr_ctl.lr)
    save_ckpt(state)                                 # last.npz = the restored weights, so a resume is safe too
    print(f'  epoch {ep} diverged ({reason}) -> best.npz (epoch {state["best_epoch"]}) restored, '
          f'lr {lr_ctl.lr:.1e}, epoch redone (retry {r + 1}/{NAN_RETRIES})')
    return True
ep = state['epoch_done'] + 1
while not state['finished'] and ep <= MAX_EPOCHS:
    est = state['last_epoch_sec'] * 1.15
    if time.time() + est > DEADLINE:
        stop_reason = f'session budget: next epoch needs ~{est/60:.0f} min'
        break
    si, K, tfr = stage_for_epoch(ep)
    counting = (si == len(CURRICULUM) - 1)            # early-stop patience counts in the last stage only
    if si != state['stage']:
        if state['stage'] >= 0:
            state['wait'], lr_ctl.wait = 0, 0
        state['stage'] = si
        print(f'== stage {si}: K={K} years per window, teacher-forcing={tfr:.2f}')

    B = global_batch_for(K)
    n_win = max(B, SAMPLES_PER_EPOCH // K)
    steps = n_win // B
    rng = np.random.default_rng([SEED, ep, state.get('nan_retries', 0)])
    tfr_t = tf.constant(tfr, tf.float32)
    t_ep, running, nb, nskip, partial = time.time(), 0.0, 0, 0, False
    pbar = tqdm(make_dataset(K, n_win, B, rng), total=steps, desc=f'epoch {ep}/{MAX_EPOCHS} K={K} B={B}')
    for env, yy, af in pbar:
        lr_ctl.on_step()
        loss, skipped = dist_step(env, yy, af, tfr_t)
        nb += 1
        if float(skipped) > 0:
            nskip += 1
        else:
            running += float(loss)
        if nb % 50 == 0 or nb == steps:
            pbar.set_postfix(loss=f'{running/max(nb - nskip, 1):.5f}', lr=f'{lr_ctl.lr:.1e}', skipped=nskip)
        if time.time() > DEADLINE:
            partial = True
            break
    pbar.close()
    # ---- divergence guard 1: non-finite steps or weights ----
    if not weights_finite() or nskip > 0.2 * max(nb, 1):
        if roll_back(f'{nskip}/{nb} non-finite steps'):
            continue
        break
    if partial:                                      # keep the progress; this epoch is redone next session
        state['train_hours'] += (time.time() - t_ep) / 3600
        save_ckpt(state)
        stop_reason = f'session budget reached mid-epoch {ep} (it will be redone on resume)'
        break

    # ---- 40-year rollout on the holdout sites: raw weights and the EMA shadow ----
    raw_w = get_tvars()
    m, _ = evaluate_holdout()
    tag, cand_w = 'raw', raw_w
    if EMA:
        ema_w = [e.numpy() for e in EMA]
        set_tvars(ema_w)
        m_ema, _ = evaluate_holdout()
        set_tvars(raw_w)                             # training always continues from the raw weights
        if m_ema[VAL_KEY][0] < m[VAL_KEY][0]:
            m, tag, cand_w = m_ema, 'ema', ema_w
    vscore, vh, va = m[VAL_KEY]
    # ---- divergence guard 2: finite but blown-up weights (holdout far worse than the best so far) ----
    if not np.isfinite(vscore) or (state['best_score'] is not None and vscore > DIVERGE_FACTOR * state['best_score']):
        if roll_back(f'holdout {vscore:.4g} vs best {state["best_score"]}'):
            continue
        break

    prev_best = np.inf if state['best_score'] is None else state['best_score']
    improved = vscore < prev_best - MIN_DELTA
    new_best = None
    if vscore < prev_best:
        state.update(best_score=float(vscore), best_epoch=ep, best_src=tag)
        new_best = cand_w
    lr_ctl.on_epoch(vscore, counting)
    if counting:
        state['wait'] = 0 if improved else state['wait'] + 1

    ep_sec = time.time() - t_ep
    state['last_epoch_sec'] = ep_sec
    state['train_hours'] += ep_sec / 3600
    state['epoch_done'] = ep
    state['history'].append(dict(epoch=ep, session=state['sessions'], K=K, tf_ratio=tfr, lr=lr_ctl.lr,
                                 train_loss=running / max(nb - nskip, 1), skipped_steps=nskip,
                                 val_metric=float(vscore),
                                 val_mse_h=vh, val_mse_agb=va, src=tag, minutes=ep_sec / 60,
                                 **{f'val_{k}': float(v[0]) for k, v in m.items()}))
    if counting and state['wait'] >= ES_PATIENCE:
        state['finished'], state['finish_kind'] = True, 'early_stop'
        stop_reason = f'early stopping (best epoch {state["best_epoch"]})'
    elif ep >= MAX_EPOCHS:
        state['finished'], state['finish_kind'] = True, 'max_epochs'
        stop_reason = 'MAX_EPOCHS reached'
    save_ckpt(state, best_weights=new_best)
    if ep in SNAPSHOT_EPOCHS:
        _atomic_savez(os.path.join(CKPT_DIR, f'snap_ep{ep}.npz'), {f'b{k}': w for k, w in enumerate(cand_w)})
    groups = ' '.join(f'{k} {v[0]:.5f}' for k, v in m.items())
    print(f'  epoch {ep}: train {running/max(nb - nskip, 1):.5f} (skipped {nskip}) | VAL[{VAL_KEY}] {vscore:.5f} ({tag}; h {vh:.4f}, '
          f'agb {va:.4f}) | {groups} | lr {lr_ctl.lr:.1e} | {ep_sec/60:.1f} min'
          + ('  *best*' if new_best is not None else ''))
    ep += 1

print('\nstopped:', stop_reason or 'loop finished')
print(f'epochs done {state["epoch_done"]}/{MAX_EPOCHS} over {state["sessions"]} session(s), '
      f'{state["train_hours"]:.2f} h of training | best VAL[{VAL_KEY}] {state["best_score"]} '
      f'(epoch {state["best_epoch"]}, {state["best_src"]})')
if not state['finished']:
    print(f'-> attach {RUN_NAME}_ckpt/ from this run\'s output and run again to continue at epoch '
          f'{state["epoch_done"] + 1}.')

best_path = os.path.join(CKPT_DIR, 'best.npz')
if os.path.isfile(best_path):
    zb = np.load(best_path)
    set_tvars([zb[f'b{i}'] for i in range(len(TVARS))])
    print('loaded best weights from', best_path)
else:
    print('no completed epoch yet -> using the current weights for the submission')
MODEL.save_weights(f'{OUT}/{RUN_NAME}.weights.h5')
hist_df = pd.DataFrame(state['history'])
hist_df.to_csv(f'{OUT}/{RUN_NAME}_history.csv', index=False)
hist_df.tail(10)

  no checkpoint found -> training from scratch
session 1: epochs done 0/6, 9.57 h of training time left in this session
== stage 0: K=1 years per window, teacher-forcing=1.00


epoch 1/6 K=1 B=1024:   0%|          | 0/1953 [00:00<?, ?it/s]

  epoch 1: train 0.01578 (skipped 0) | VAL[shift] 0.03907 (raw; h 0.0507, agb 0.0274) | shift 0.03907 random 0.08989 all 0.05940 | lr 3.0e-04 | 36.1 min  *best*


epoch 2/6 K=1 B=1024:   0%|          | 0/1953 [00:00<?, ?it/s]

  epoch 2: train 0.01838 (skipped 0) | VAL[shift] 0.03923 (ema; h 0.0557, agb 0.0227) | shift 0.03923 random 0.08033 all 0.05567 | lr 3.0e-04 | 35.7 min


epoch 3/6 K=1 B=1024:   0%|          | 0/1953 [00:00<?, ?it/s]

... (4 outputs left out)

    ReduceLROnPlateau -> lr = 1.50e-04
  epoch 5: train 0.01757 (skipped 0) | VAL[shift] 0.03830 (ema; h 0.0499, agb 0.0267) | shift 0.03830 random 0.09762 all 0.06203 | lr 1.5e-04 | 35.7 min


epoch 6/6 K=1 B=1024:   0%|          | 0/1953 [00:00<?, ?it/s]

  epoch 6: train 0.01537 (skipped 0) | VAL[shift] 0.03853 (ema; h 0.0423, agb 0.0348) | shift 0.03853 random 0.06616 all 0.04958 | lr 1.5e-04 | 35.7 min

stopped: MAX_EPOCHS reached
epochs done 6/6 over 1 session(s), 3.57 h of training | best VAL[shift] 0.032069901935756207 (epoch 3, ema)
loaded best weights from /kaggle/working/global_noco2_s7_ckpt/best.npz


,epoch,session,K,tf_ratio,lr,train_loss,skipped_steps,val_metric,val_mse_h,val_mse_agb,src,minutes,val_shift,val_random,val_all
0,1,1,1,1.0,0.00030,0.015776,0,0.039067,0.050717,0.027417,raw,36.079033,0.039067,0.089891,0.059396
1,2,1,1,1.0,0.00030,0.018381,0,0.039234,0.055745,0.022722,ema,35.663915,0.039234,0.080334,0.055674
2,3,1,1,1.0,0.00030,0.016826,0,0.032070,0.047166,0.016973,ema,35.665297,0.032070,0.071405,0.047804
3,4,1,1,1.0,0.00030,0.017932,0,0.034555,0.045592,0.023518,ema,35.672655,0.034555,0.060569,0.044961
4,5,1,1,1.0,0.00015,0.017571,0,0.038304,0.049926,0.026682,ema,35.674671,0.038304,0.097621,0.062031
5,6,1,1,1.0,0.00015,0.015371,0,0.038528,0.042292,0.034764,ema,35.673079,0.038528,0.066157,0.049579


## 10. Holdout evaluation with the best weights

In [13]:
m_best, hold_pred = evaluate_holdout(desc='holdout rollout')
for k, (sc, h, a) in m_best.items():
    print(f'holdout {k:6s}: metric {sc:.5f} (height {h:.5f}, agb {a:.5f}) | {len(HOLD_GROUPS[k])} sites')
rows = [dict(group=k, age=age, metric=scored_metric(hold_pred[ai:ai + 1, idx], hold_y40[ai:ai + 1, idx])[0])
        for k, idx in HOLD_GROUPS.items() if k != 'all' for ai, age in enumerate(SEED_AGES)]
pd.DataFrame(rows).pivot(index='age', columns='group', values='metric').round(4)

holdout rollout:   0%|          | 0/40 [00:00<?, ?it/s]

holdout shift : metric 0.03207 (height 0.04717, agb 0.01697) | 1500 sites
holdout random: metric 0.07141 (height 0.06154, agb 0.08127) | 1000 sites
holdout all   : metric 0.04780 (height 0.05291, agb 0.04269) | 2500 sites


group,random,shift
age,,
1,0.0462,0.0228
10,0.0398,0.0166
20,0.0461,0.0204
30,0.0595,0.0169
50,0.0720,0.0311
70,0.1009,0.0516
100,0.0833,0.0236
150,0.0770,0.0319
200,0.0769,0.0367


## 11. Test rollout & submissions (written every session)

For the best-holdout weights and every snapshot epoch (`SNAPSHOT_EPOCHS`), one 40-year rollout per variant:

* `none`: the test forcing as given (the only variant with `CO2_MODE='const'`).
* `co2_hist`: CO2 block replaced by the grid's mean 1980-2020 series.
* `co2x<f>`: CO2 block moved f times as far toward that series (`co2x1` = `co2_hist`).

Files: `submission.csv` (best weights, first variant), `submission_<variant>.csv`, `submission_ep<k>_<variant>.csv`.
`<RUN_NAME>_preds.npz` holds every rollout for `ensemble.py`.

In [14]:
def load_test(scn):
    d = np.load(f'{COMP}/test_{scn}.npz')
    return harmonize(d['test_x'].astype(np.float32)), d['test_y0'].astype(np.float32)

def test_env(tx, variant):
    """'none' | 'clip' | 'co2_hist' | 'co2x<f>' (CO2 block = test + f * (grid series - test); co2x1 == co2_hist)."""
    f = 1.0 if variant == 'co2_hist' else (float(variant[4:]) if variant.startswith('co2x') else None)
    def env_at(t):
        e = tx[:, t]
        if variant == 'clip':
            e = np.clip(e, XMIN_G, XMAX_G)
        elif f is not None:
            e = e.copy(); e[..., CO2_FEATS] += f * (CO2_SERIES[t] - e[..., CO2_FEATS])
        z = np.clip(normData(e, X_MEAN, X_STD), -Z_CLIP, Z_CLIP)
        if CO2_MODE == 'const':
            z[..., CO2_FEATS] = 0.0
        return z.astype(np.float32)
    return env_at

def write_submission(p_by_scn, path):
    parts = []
    for scn in SCENARIOS:
        p = p_by_scn[scn]; N = p.shape[1]
        ids = [f'{scn}_{site}_{age}' for age in SEED_AGES for site in range(N)]
        parts.append(pd.DataFrame({'id': ids,
                                   'height': (p[:, :, 0] / Y_MEAN[0]).reshape(-1),
                                   'agb':    (p[:, :, 1] / Y_MEAN[1]).reshape(-1)}))
    sub = sample_sub[['id']].join(pd.concat(parts).set_index('id'), on='id')
    assert len(sub) == len(sample_sub)
    assert not sub[['height', 'agb']].isna().any().any(), 'NaNs / missing ids!'
    sub.to_csv(path, index=False)
    return sub

def msd(a, b):
    return 0.5 * (np.mean(((a[..., 0] - b[..., 0]) / Y_MEAN[0]) ** 2) + np.mean(((a[..., 1] - b[..., 1]) / Y_MEAN[1]) ** 2))

# weight sets: the best holdout epoch (already loaded) + every saved snapshot epoch
weight_sets = [('best', get_tvars())]
for p in sorted(glob.glob(os.path.join(CKPT_DIR, 'snap_ep*.npz')), key=lambda s: int(s.split('snap_ep')[1][:-4])):
    if int(os.path.basename(p)[7:-4]) == int(state['best_epoch'] or -1):
        continue                                     # same weights as 'best'
    zs = np.load(p)
    weight_sets.append((os.path.basename(p)[5:-4], [zs[f'b{k}'] for k in range(len(TVARS))]))
print('test weight sets:', [n for n, _ in weight_sets], f'(best = epoch {state["best_epoch"]}) | variants', TEST_VARIANTS)

sample_sub = pd.read_csv(f'{COMP}/sample_submission.csv')
TEST = {scn: load_test(scn) for scn in SCENARIOS}
preds, rows = {}, []
for wname, ws in weight_sets:
    set_tvars(ws)
    for v in TEST_VARIANTS:
        key = v if wname == 'best' else f'{wname}_{v}'
        preds[key] = {scn: rollout(test_env(tx, v), tx.shape[0], np.transpose(ty0, (1, 0, 2)), desc=f'{scn} {key}')
                      for scn, (tx, ty0) in TEST.items()}
        path = (f'{OUT}/submission.csv' if key == TEST_VARIANTS[0] else f'{OUT}/submission_{key}.csv')
        sub = write_submission(preds[key], path)
        both = np.concatenate([preds[key][s] for s in SCENARIOS], axis=1)
        rows.append(dict(file=os.path.basename(path), weights=wname, variant=v,
                         height=sub.height.mean(), agb=sub.agb.mean()))
        print(f'wrote {path} | height mean {sub.height.mean():.3f}, agb mean {sub.agb.mean():.3f}')
set_tvars(weight_sets[0][1])
del TEST; gc.collect()

ref = preds[TEST_VARIANTS[0]]
for r in rows:
    k = r['variant'] if r['weights'] == 'best' else f"{r['weights']}_{r['variant']}"
    r['msd_vs_first'] = float(np.mean([msd(preds[k][s], ref[s]) for s in SCENARIOS]))
print(pd.DataFrame(rows).round(4).to_string(index=False))

np.savez_compressed(f'{OUT}/{RUN_NAME}_preds.npz',
                    **{f'{scn}_{key}': p for key, d in preds.items() for scn, p in d.items()},
                    holdout_pred=hold_pred, holdout_y40=hold_y40, holdout_sites=HOLD_SITES)
print('wrote', f'{OUT}/{RUN_NAME}_preds.npz', '| keys <scenario>_<weights>_<variant>, best weights without prefix')
print(f'total session time {(time.time() - NB_T0) / 3600:.2f} h')

test weight sets: ['best', 'ep2', 'ep4', 'ep5'] (best = epoch 3) | variants ('none',)


ssp126 none:   0%|          | 0/40 [00:00<?, ?it/s]

ssp585 none:   0%|          | 0/40 [00:00<?, ?it/s]

wrote /kaggle/working/submission.csv | height mean 0.955, agb mean 0.963


ssp126 ep2_none:   0%|          | 0/40 [00:00<?, ?it/s]

ssp585 ep2_none:   0%|          | 0/40 [00:00<?, ?it/s]

... (3 outputs left out)

wrote /kaggle/working/submission_ep4_none.csv | height mean 0.963, agb mean 0.928


ssp126 ep5_none:   0%|          | 0/40 [00:00<?, ?it/s]

ssp585 ep5_none:   0%|          | 0/40 [00:00<?, ?it/s]

wrote /kaggle/working/submission_ep5_none.csv | height mean 0.978, agb mean 0.959
                   file weights variant  height    agb  msd_vs_first
         submission.csv    best    none  0.9546 0.9629        0.0000
submission_ep2_none.csv     ep2    none  0.9647 0.9663        0.0258
submission_ep4_none.csv     ep4    none  0.9630 0.9278        0.0217
submission_ep5_none.csv     ep5    none  0.9776 0.9587        0.0567
wrote /kaggle/working/global_noco2_s7_preds.npz | keys <scenario>_<weights>_<variant>, best weights without prefix
total session time 5.74 h


## 12. Notes

* `global_lstmxl_preds.npz` has `ssp126`/`ssp585` (variant `none`), `<scn>_clip`, `<scn>_co2_hist`, and the
  holdout predictions. It has no `val_pred` on the old 852 val sites, so blend it with the LB-based weights of
  `ensemble.py` (the VAL-based comparison there needs `val_pred` from every run).
* `global_lstmxl_holdout.npz` lists the holdout sites (and the earlier runs' training sites), so older models can be
  scored on exactly the same sites to check whether the shift holdout ranks models the way the LB does.
* Lighter: `PRESET='lite'`, lower `SAMPLES_PER_EPOCH`, or cap the curriculum at K=8. If K≥4 runs out of GPU memory,
  lower `ROLL_TOKENS` (allowed between sessions).
* The predictions share the LB-measured AGB bias (ssp126 ~x1.02, ssp585 ~x1.12 for the old blend): re-estimate the
  per-scenario factors for this model with `probe_bias.py` before relying on its raw score.
* `HARMONIZE=True` lowers the holdout score a little (the holdout truth still depends on the interannual variability
  the model no longer sees); that is the price of matching the test inputs, which never contain it.
* Another backbone = same notebook with `BACKBONE='ecoformer'` or `'fusion'` and another `RUN_NAME`.